# Demo 04 : Handoffs

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We use the made-up Veenhaven Bus records. We count every attempted model call, input and output tokens, and elapsed seconds, including the shared rate limiter.

In [2]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import NotRequired, TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, Send
from pydantic import BaseModel, Field


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1200, max_retries=0)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(pattern=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, seconds=round(time.perf_counter() - start, 1),
                   malformed=counter.invalid, errors=counter.errors)
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def part_stock(part: str) -> str:
    """Look up the stock, location and name of a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the manual's meaning and first action for a Veenhaven Bus error code."""
    matches = [f"[{name}]\n{line}" for name, text in PAGES.items()
               for line in text.splitlines() if line.startswith("| " + error_code.upper() + " |")]
    return "\n".join(matches) or "No matching error code."


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: buses(fleet_no, model, depot_id, commissioned, firmware),
    depots(depot_id, name, city, contract), parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, fleet_no, opened, status, error_code, summary, mechanic_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(30))


@tool
def search_manuals(query: str) -> str:
    """Find relevant Veenhaven Bus manuals and bulletins with their file names."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:3]
    return "\n\n".join(f"[{name}]\n{PAGES[name]}" for name in ranked)


QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition for this error? Be brief.")
STOCK_TASK = "How many CE-CF-02 cooling fans are in stock and at which location?"
MANUAL_TASK = "What are the meaning, first action and replacement condition for F-CE-110? Cite the manual."
LIMIT = {"recursion_limit": 24}


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def agent_by_hand(model, tools, prompt, state_schema=MessagesState):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(state_schema)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def specialists(model):
    return {
        "parts": agent_by_hand(model, [part_stock], "Answer stock questions from the tool. Return every fact briefly."),
        "manuals": agent_by_hand(model, [fault_guidance], "Answer error-code questions from the manual tool. Cite its file."),
    }


def compose(model, question, findings):
    return model.invoke("Answer from these findings only. Preserve counts, locations and conditions.\n"
                        + question + "\n" + "\n".join(findings)).content


def completeness(answer):
    text = answer.lower()
    return {"stock": bool(re.search(r"\b(6|six)\b", text)) and "rietpoort" in text,
            "action": "fan" in text and "airflow" in text and bool(re.search(r"\b(80|eighty)\b", text))}

## a · handoffs

We adapt the documented subgraph transfer, which forwards the last AI message and its transfer receipt. We seed the triggering tool calls so the two faults remain reproducible. The receiver still calls the live course model. Source : [LangChain handoffs](https://docs.langchain.com/oss/python/langchain/multi-agent/handoffs).

In [3]:
class HandoffState(AgentState):
    active_agent: NotRequired[str]


refused = []


def transfer(target, fixed):
    @tool("transfer_to_" + target)
    def move(runtime: ToolRuntime) -> Command | str:
        """Transfer the conversation to the other specialist. Call it on its own, after your lookups."""
        messages = runtime.state["messages"]
        if fixed and len(messages[-1].tool_calls) > 1:
            refused.append(target)
            return "Transfer refused : call the transfer on its own, after the other tool results."
        note = ToolMessage(content="Transferred to " + target, tool_call_id=runtime.tool_call_id)
        forwarded = messages if fixed else [next(m for m in reversed(messages) if isinstance(m, AIMessage))]
        return Command(goto=target, graph=Command.PARENT,
                       update={"active_agent": target, "messages": [*forwarded, note]})
    return move


def handoff_graph(model, fixed=True):
    parts = agent_by_hand(model, [part_stock, transfer("manuals_agent", fixed)],
                          "Look up the stock first. Then transfer to manuals_agent if the error guidance is unanswered.", HandoffState)
    manuals = agent_by_hand(model, [fault_guidance, transfer("parts_agent", fixed)],
                            "Look up error guidance. Transfer only if stock is still unanswered. "
                            "When both facts are present, give one complete brief answer.", HandoffState)

    def route(state):
        last = state["messages"][-1]
        return END if isinstance(last, AIMessage) and not last.tool_calls else state["active_agent"]

    graph = StateGraph(HandoffState)
    graph.add_node("parts_agent", lambda s: parts.invoke(s, LIMIT))
    graph.add_node("manuals_agent", lambda s: manuals.invoke(s, LIMIT))
    graph.add_conditional_edges(START, lambda s: s.get("active_agent", "parts_agent"), ["parts_agent", "manuals_agent"])
    for name in ["parts_agent", "manuals_agent"]:
        graph.add_conditional_edges(name, route, ["parts_agent", "manuals_agent", END])
    return graph.compile()


def seeded_failure(parallel, fixed=False):
    stock_call = AIMessage(content="", tool_calls=[{"name": "part_stock", "args": {"part": "CE-CF-02"}, "id": "stock-1", "type": "tool_call"}])
    move_call = {"name": "transfer_to_manuals_agent", "args": {}, "id": "move-1", "type": "tool_call"}
    trigger = AIMessage(content="", tool_calls=[*stock_call.tool_calls, move_call] if parallel else [move_call])
    stock_result = ToolMessage(content=part_stock.invoke({"part": "CE-CF-02"}), tool_call_id="stock-1")

    def seed(state):
        return {"messages": [trigger] if parallel else [stock_call, stock_result, trigger]}

    child = StateGraph(HandoffState)
    child.add_node("seed", seed)
    child.add_node("tools", ToolNode([part_stock, transfer("manuals_agent", fixed)]))
    child.add_edge(START, "seed")
    child.add_edge("seed", "tools")
    child.add_edge("tools", END)

    def receive(state):
        print("received :", [(m.type, str(m.content)[:140]) for m in state["messages"]])
        response = llm.invoke([SystemMessage("Answer using only the visible facts and the manual excerpt. "
                                            "If the stock count is missing, say it is unavailable.\n"
                                            + fault_guidance.invoke({"error_code": "F-CE-110"})), *state["messages"]])
        return {"messages": [response]}

    parent = StateGraph(HandoffState)
    parent.add_node("parts_agent", child.compile())
    parent.add_node("manuals_agent", receive)
    parent.add_edge(START, "parts_agent")
    parent.add_edge("manuals_agent", END)
    return parent.compile().invoke(ask(QUESTION), LIMIT)

We first put a stock lookup and a transfer in the same AI message. The jump forwards both tool requests but only the transfer receipt. We catch the provider's rejection so Run All continues.

In [4]:
def missing_sibling():
    try:
        result = seeded_failure(parallel=True)
        print("provider accepted :", last_text(result))
    except Exception as exc:
        if "No tool output found" not in str(exc) and "tool_call_ids" not in str(exc):
            raise
        print("provider error :", type(exc).__name__, str(exc)[:900])


measured("handoff missing sibling", missing_sibling)

received : [('human', "Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, at which location, and what are the manual's first action an"), ('ai', ''), ('tool', 'Transferred to manuals_agent')]


provider error : OpenAIInvalidRequestError Error code: 400 - {'error': {'message': 'Provider returned error', 'code': 400, 'metadata': {'raw': '{\n  "error": {\n    "message": "No tool output found for function call stock-1.",\n    "type": "invalid_request_error",\n    "param": "input",\n    "code": null\n  }\n}', 'provider_name': 'Azure', 'is_byok': False, 'previous_errors': [{'code': 400, 'message': 'Provider returned error', 'provider_name': 'OpenAI', 'raw': '{\n  "error": {\n    "message": "No tool output found for function call stock-1.",\n    "type": "invalid_request_error",\n    "param": "input",\n    "code": null\n  }\n}'}]}}, 'user_id': 'user_3KeVpEuQwLfvCui96GJ2WLdwfCV'}
metrics : {"pattern": "handoff missing sibling", "calls": 1, "input_tokens": 0, "output_tokens": 0, "seconds": 3.3, "malformed": 0, "errors": 1}


We then separate the lookup from the transfer. The history is valid, but the last-message transfer loses the earlier stock result. We read the half answer and check the missing fact.

In [5]:
result = measured("handoff lost work", lambda: seeded_failure(parallel=False))
print("half answer :", last_text(result))
print("checks :", completeness(last_text(result)))

received : [('human', "Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, at which location, and what are the manual's first action an"), ('ai', ''), ('tool', 'Transferred to manuals_agent')]


metrics : {"pattern": "handoff lost work", "calls": 1, "input_tokens": 156, "output_tokens": 40, "seconds": 3.1, "malformed": 0, "errors": 0}
half answer : Stock count and location for CE-CF-02 are unavailable. For F-CE-110, first check the traction cooling fan; replace it if airflow is below 80%.
checks : {'stock': False, 'action': True}


## b · handoffs-fixed

We fix both faults in the transfer tool. It refuses to transfer when the same AI message holds other tool calls, so every sibling call gets its result and the model can transfer on its next turn. It forwards the full worker history with the transfer receipt. The API flag `parallel_tool_calls=False` would be the other route, but OpenRouter drops it for the course model, so we enforce the rule in code.

We replay the seeded parallel turn through the fixed tool first. No model call is needed : the stock result and the refusal both answer their tool calls, and nothing jumps.

In [6]:
result = seeded_failure(parallel=True, fixed=True)
print("history :", [(m.type, str(m.content)[:80]) for m in result["messages"]])
print("refused :", refused)

history : [('human', 'Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, '), ('ai', ''), ('tool', '[["CE-CF-02", "traction cooling fan", 6, "Rietpoort"]]'), ('tool', 'Transfer refused : call the transfer on its own, after the other tool results.')]
refused : ['manuals_agent']


We now let both agents choose their tools live. A valid history still does not guarantee that the parts agent transfers.

In [7]:
refused.clear()
result = measured("handoffs fixed", lambda: handoff_graph(llm).invoke(ask(QUESTION), LIMIT))
print("refused transfers :", len(refused))
print("tools :", [m.name for m in result["messages"] if isinstance(m, ToolMessage)])
print("answer :", last_text(result))
print("checks :", completeness(last_text(result)))

metrics : {"pattern": "handoffs fixed", "calls": 4, "input_tokens": 947, "output_tokens": 116, "seconds": 10.4, "malformed": 0, "errors": 0}
refused transfers : 0
tools : ['part_stock', 'transfer_to_manuals_agent', 'fault_guidance']
answer : There are **6 CE-CF-02 cooling fans** in stock at **Rietpoort**. For F-CE-110, **check the traction cooling fan**; replace it if airflow is **below 80%**.
checks : {'stock': True, 'action': True}


## c · handoffs-one-agent

We keep one conversation and change its prompt and tools from the active step in state. We preserve history naturally inside the agent, while middleware controls which tools each step sees.

In [8]:
class SupportState(AgentState):
    current_step: str
    bus: str


@tool
def record_bus(fleet_no: str, runtime: ToolRuntime[None, SupportState]) -> Command:
    """Record the fleet number supplied by the caller and move to the specialist step."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        exists = db.execute("SELECT 1 FROM buses WHERE fleet_no = ?", (fleet_no,)).fetchone()
    if not exists:
        return Command(update={"messages": [ToolMessage(content="Unknown fleet number. Ask for a valid number.", tool_call_id=runtime.tool_call_id)]})
    return Command(update={"bus": fleet_no, "current_step": "specialist",
                           "messages": [ToolMessage(content="Recorded " + fleet_no, tool_call_id=runtime.tool_call_id)]})


@wrap_model_call
def apply_step(request, handler):
    step = request.state.get("current_step", "triage")
    print("step :", step)
    if step == "triage":
        return handler(request.override(system_prompt="Collect the fleet number, then call record_bus. Do not give technical advice.", tools=[record_bus]))
    return handler(request.override(system_prompt="Answer briefly for bus " + request.state["bus"] + " using the tools.", tools=[part_stock, fault_guidance]))


from langgraph.checkpoint.memory import InMemorySaver

support = create_agent(llm, tools=[record_bus, part_stock, fault_guidance],
                       state_schema=SupportState, middleware=[apply_step], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "bus-support"}, "recursion_limit": 24}
result = measured("triage", lambda: support.invoke({**ask("My bus shows F-CE-110."), "current_step": "triage", "bus": ""}, config))
print("answer :", last_text(result))
result = measured("specialist step", lambda: support.invoke(ask(QUESTION), config))
print("active :", result["current_step"], result["bus"])
print("answer :", last_text(result))
print("checks :", completeness(last_text(result)))

step : triage


step : triage


metrics : {"pattern": "triage", "calls": 2, "input_tokens": 206, "output_tokens": 54, "seconds": 4.7, "malformed": 0, "errors": 0}
answer : I couldn’t match **F-CE-110** to a fleet number. Could you check it and provide a valid fleet number?
step : triage


step : specialist


step : specialist


metrics : {"pattern": "specialist step", "calls": 3, "input_tokens": 938, "output_tokens": 142, "seconds": 8.7, "malformed": 0, "errors": 0}
active : specialist CE12-2021-0117
answer : **F-CE-110** means traction system over-temperature. First, check the traction cooling fan; replace it if airflow is below **80%**. **CE-CF-02** fans: **6 in stock**, at **Rietpoort**.
checks : {'stock': True, 'action': True}


We inspect triage failures as well. A model may mistake an error code for a fleet number ; the database check prevents that value from advancing the step. We keep current_step unchanged until a valid fleet number is recorded.

## Things to try

Optional : we can change the question, inspect each context, and compare the answer checks with the call and token counts.